In [1]:
import os, random, shutil
import numpy as np
import pandas as pd
from glob import glob
from collections import defaultdict
from scipy.ndimage import label, find_objects
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
from skimage import measure
from sklearn.model_selection import train_test_split
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import segmentation_models_pytorch as smp
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
%matplotlib inline
 
IMAGES_ROOT = "../data/20_trees/Images"
GT_ROOT     = "../data/20_trees/Ground_truths"
CACHE_DIR   = "./eval_cache"
OUT_DIR     = "./outputs/evaluation_3d"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)
 
CLASS_REMAP    = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
CLASS_NAMES    = ["Background", "Bark", "Wood", "Knot", "Crack"]
NUM_CLASSES    = 5
DEFECT_CLASSES = [3, 4]
CLASS_COLORS_PLOTLY = {
    0: "rgb(74,85,104)",   1: "rgb(141,110,99)",
    2: "rgb(212,169,106)", 3: "rgb(255,111,0)",
    4: "rgb(229,57,53)",
}
 
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
MODELS = {
    "DilatedCNN+MRF log-split": {
        "path": "../logs/20_trees/checkpoints/best.pt",
        "type": "dilated",
        "mrf":  True,
        "mrf_beta": 0.8,
    },
    "DilatedCNN+MRF random-split": {
        "path": "../logs/20_trees_random-split/checkpoints/best.pt",
        "type": "dilated",
        "mrf":  True,
        "mrf_beta": 0.8,
    },
    "UNet++ log-split": {
        "path": "../logs/20_trees_UNET++/checkpoints/best.pt",
        "type": "unetpp",
        "mrf":  False,
        "mrf_beta": 0.0,
    },
    "UNet++ random-split": {
        "path": "../logs/20_trees_UNET++_random-split/checkpoints/best.pt",
        "type": "unetpp",
        "mrf":  False,
        "mrf_beta": 0.0,
    },
}

# Conditions to compare - 2D only vs 2D + 3D MRF
EVAL_CONDITIONS = {
    "2D only":     False,
    "2D + 3D MRF": True,
}
 
# Common test logs - held out across all training runs
COMMON_TEST_LOGS = ["Dub 9", "kmen9"]
 
# Boundary metrics only for defect classes with fewer voxels than this
# Background/Wood have millions of voxels - directed_hausdorff would run for hours
MAX_BOUNDARY_VOXELS = 50_000

In [3]:
PREFIX_OVERRIDE = {}
 
def mask_stem_to_image_stem(mask_filename):
    name  = os.path.splitext(mask_filename)[0]
    parts = name.split("_", 2)
    stem  = parts[2] if len(parts) == 3 else name
    for wrong, correct in PREFIX_OVERRIDE.items():
        if stem.startswith(wrong):
            stem = correct + stem[len(wrong):]
            break
    return stem
 
image_paths  = sorted(glob(os.path.join(IMAGES_ROOT, "**", "*.tif"), recursive=True))
mask_paths   = sorted(glob(
    os.path.join(GT_ROOT, "**", "GroundTruthProject", "PixelLabelData", "*.png"),
    recursive=True))
image_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
 
all_pairs = []
for mp in mask_paths:
    stem = mask_stem_to_image_stem(os.path.basename(mp))
    if stem in image_lookup:
        all_pairs.append((image_lookup[stem], mp))
 
def get_log_id(img_path):
    parts     = img_path.replace("\\", "/").split("/")
    log_names = set(os.listdir(IMAGES_ROOT))
    return next((p for p in parts if p in log_names), None)
 
log_pairs = defaultdict(list)
for pair in all_pairs:
    log_id = get_log_id(pair[0])
    if log_id:
        log_pairs[log_id].append(pair)
 
all_logs   = sorted(log_pairs.keys())
test_pairs = [p for lg in COMMON_TEST_LOGS for p in log_pairs[lg]]
print(f"Total pairs: {len(all_pairs)} | Test pairs: {len(test_pairs)}")

Total pairs: 1347 | Test pairs: 128


In [4]:
PATCH_SIZE = 256
 
def crop_to_foreground(img, mask, margin=10):
    coords = np.column_stack(np.where(mask > 0))
    if coords.size == 0:
        return img, mask
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    return (img[max(y_min-margin,0):min(y_max+margin,img.shape[0]),
                max(x_min-margin,0):min(x_max+margin,img.shape[1])],
            mask[max(y_min-margin,0):min(y_max+margin,mask.shape[0]),
                 max(x_min-margin,0):min(x_max+margin,mask.shape[1])])
 
def remap_mask(mask):
    out = np.zeros_like(mask, dtype=np.int64)
    for raw, idx in CLASS_REMAP.items():
        out[mask == raw] = idx
    return out
 
def load_and_preprocess(img_path, mask_path):
    img  = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if mask.shape[:2] != img.shape[:2]:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
    img, mask = crop_to_foreground(img, mask)
    img  = cv2.resize(img,  (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)
    img  = img.astype(np.float32)
    img  = (img - img.mean()) / (img.std() + 1e-6)
    img  = np.expand_dims(img, axis=0)
    return np.array(img, dtype=np.float32), np.array(remap_mask(mask), dtype=np.int64)
 
def build_cache(pairs, cache_dir, force=False):
    if force and os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
    os.makedirs(cache_dir, exist_ok=True)
    cached = []
    for img_path, mask_path in tqdm(pairs, desc="Caching"):
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        img_file  = os.path.join(cache_dir, stem + "_img.npy")
        mask_file = os.path.join(cache_dir, stem + "_mask.npy")
        if not os.path.exists(img_file):
            img, mask = load_and_preprocess(img_path, mask_path)
            np.save(img_file, img); np.save(mask_file, mask)
        cached.append((img_file, mask_file))
    print(f"  Cache ready: {len(cached)} pairs")
    return cached
 
cached_test = build_cache(test_pairs, os.path.join(CACHE_DIR, "test"), force=False)
 

Caching: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 5119.93it/s]

  Cache ready: 128 pairs


In [5]:
class DilatedSegCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=NUM_CLASSES):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = self._block(in_channels, 64,  dilation=1)
        self.enc2 = self._block(64,  128, dilation=2)
        self.enc3 = self._block(128, 256, dilation=4)
        self.enc4 = self._block(256, 512, dilation=8)
        self.dec3 = self._upblock(512, 256)
        self.dec2 = self._upblock(256, 128)
        self.dec1 = self._upblock(128, 64)
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=1)
    def _block(self, in_c, out_c, dilation):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def _upblock(self, in_c, out_c):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def forward(self, x):
        x1=self.enc1(x); x2=self.enc2(self.pool(x1))
        x3=self.enc3(self.pool(x2)); x4=self.enc4(self.pool(x3))
        x=self.dec3(x4)+x3; x=self.dec2(x)+x2; x=self.dec1(x)+x1
        return self.classifier(x)
 
def build_unetpp():
    model = smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights=None,
                              in_channels=1, classes=NUM_CLASSES, activation=None)
    enc_w = tvm.resnet34(weights=tvm.ResNet34_Weights.IMAGENET1K_V1).state_dict()
    enc_w["conv1.weight"] = enc_w["conv1.weight"].mean(dim=1, keepdim=True)
    model_dict = model.encoder.state_dict()
    matched    = {k: v for k, v in enc_w.items()
                  if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(matched)
    model.encoder.load_state_dict(model_dict, strict=False)
    print(f"  UNet++ encoder: {len(matched)}/{len(model_dict)} weights loaded")
    return model
 
def load_checkpoint(model_cfg):
    model = DilatedSegCNN() if model_cfg["type"] == "dilated" else build_unetpp()
    model.load_state_dict(torch.load(model_cfg["path"], map_location=device))
    return model.to(device).eval()

In [6]:
def mrf_gibbs_sampling_2d(prob_map, iterations=5, beta=0.8):
    C, H, W = prob_map.shape; dev = prob_map.device
    labels  = torch.argmax(prob_map, dim=0)
    kernel  = torch.ones((C, 1, 3, 3), device=dev); kernel[:,:,1,1] = 0
    for _ in range(iterations):
        onehot = F.one_hot(labels, num_classes=C).permute(2,0,1)[None].float()
        nc     = F.conv2d(onehot, kernel, padding=1, groups=C).squeeze(0)
        energy = -torch.log(prob_map + 1e-6) + beta * (8 - nc)
        labels = torch.argmin(energy, dim=0)
    return labels
 
def mrf_gibbs_sampling_3d(prob_map, iterations=3, beta=0.3,
                          skip_classes=None):
    """
    Vectorized 3D MRF - single grouped conv3d per iteration.
    Adapted from mrf3d.py.
    beta=0.3 - soft smoothing to preserve thin crack structures.
    skip_classes: list of class indices to freeze (e.g. [4] to skip Crack).
    After refinement, frozen class voxels are restored from original argmax.
    """
    if skip_classes is None:
        skip_classes = [4]   # skip Crack by default - degrades on crack-heavy logs
 
    C, D, H, W = prob_map.shape; dev = prob_map.device
    original_labels = torch.argmax(prob_map, dim=0)   # keep for restoring frozen classes
    labels          = original_labels.clone()
    kernel          = torch.ones((C, 1, 3, 3, 3), device=dev)
    kernel[:,:,1,1,1] = 0
 
    for _ in range(iterations):
        # one_hot: (D,H,W) -> (D,H,W,C) -> (1,C,D,H,W)
        onehot = F.one_hot(labels, num_classes=C).permute(3,0,1,2).float().unsqueeze(0)
        nc     = F.conv3d(onehot, kernel, padding=1, groups=C).squeeze(0)
        energy = -torch.log(prob_map + 1e-6) + beta * (26 - nc)
        labels = torch.argmin(energy, dim=0)
 
    # restore frozen classes - do not let MRF vote them away
    for cls in skip_classes:
        mask           = (original_labels == cls)
        labels[mask]   = cls
 
    return labels

In [7]:
def build_log_volumes(model_cfg, apply_3d_mrf=False):
    """
    Returns:
      gt_vols:   {log_id: (D,H,W) int64}   ground truth
      pred_vols: {log_id: (D,H,W) int64}   predictions
      prob_vols: {log_id: (C,D,H,W) float} softmax probabilities
    """
    model   = load_checkpoint(model_cfg)
    use_mrf = model_cfg["mrf"]
    beta    = model_cfg.get("mrf_beta", 0.8)
 
    log_to_indices = defaultdict(list)
    for i, pair in enumerate(test_pairs):
        log_to_indices[get_log_id(pair[0])].append(i)
 
    gt_vols = {}; pred_vols = {}; prob_vols = {}
 
    for log_id, indices in log_to_indices.items():
        imgs  = np.stack([np.load(cached_test[i][0]) for i in indices])
        masks = np.stack([np.load(cached_test[i][1]) for i in indices])
        loader = DataLoader(
            TensorDataset(torch.tensor(imgs, dtype=torch.float32),
                          torch.tensor(masks, dtype=torch.long)),
            batch_size=16, shuffle=False, num_workers=0)
 
        slice_probs = []; slice_preds = []
        with torch.no_grad():
            for imgs_b, _ in tqdm(loader, desc=f"  {log_id}", leave=False):
                probs_b = F.softmax(model(imgs_b.to(device)), dim=1)
                if use_mrf:
                    preds_b = torch.stack([
                        mrf_gibbs_sampling_2d(probs_b[b], beta=beta)
                        for b in range(len(probs_b))]).cpu()
                else:
                    preds_b = torch.argmax(probs_b, dim=1).cpu()
                slice_probs.append(probs_b.cpu())
                slice_preds.append(preds_b)
 
        pred_vol = torch.cat(slice_preds, dim=0).numpy()          # (D,H,W)
        prob_vol = torch.cat(slice_probs, dim=0).numpy()           # (D,C,H,W)
        prob_vol = prob_vol.transpose(1, 0, 2, 3)                  # (C,D,H,W)
 
        if apply_3d_mrf:
            print(f"    Applying 3D MRF on {log_id} "
                  f"(volume shape: {prob_vol.shape})...")
            prob_tensor = torch.tensor(prob_vol, dtype=torch.float32, device=device)
            pred_vol    = mrf_gibbs_sampling_3d(prob_tensor).cpu().numpy()
 
        gt_vols[log_id]   = masks
        pred_vols[log_id] = pred_vol
        prob_vols[log_id] = prob_vol
 
    del model; torch.cuda.empty_cache()
    return gt_vols, pred_vols, prob_vols

In [8]:
def safe_compactness(region_mask):
    try:
        verts, _, _, _ = measure.marching_cubes(
            region_mask.astype(np.float32), level=0.5)
        return (float(verts.shape[0])**1.5) / (float(np.count_nonzero(region_mask))+1e-6)
    except:
        return None
 
def compute_3d_boundary_metrics(pred_bin, gt_bin):
    """3D HD and ASSD on voxel coordinates."""
    pred_pts = np.column_stack(np.where(pred_bin))
    gt_pts   = np.column_stack(np.where(gt_bin))
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float("nan"), float("nan")
    hd   = max(directed_hausdorff(pred_pts, gt_pts)[0],
               directed_hausdorff(gt_pts, pred_pts)[0])
    assd = (cKDTree(gt_pts).query(pred_pts)[0].mean() +
            cKDTree(pred_pts).query(gt_pts)[0].mean()) / 2
    return hd, assd
 
def compute_volume_metrics_3d(pred_vol, gt_vol, class_idx):
    """
    Full 3D metric suite - overlap + boundary + volume structure.
 
    IMPORTANT - annotation mismatch caveat:
    GT was annotated slice-by-slice in 2D without enforcing 3D consistency.
    Overlap metrics measure agreement with 2D annotation decisions.
    Volume metrics measure anatomical plausibility of the 3D structure.
    These two sets measure fundamentally different things - report both.
    """
    pred_bin = (pred_vol == class_idx)
    gt_bin   = (gt_vol   == class_idx)
 
    # global overlap metrics (matches training notebook) 
    tp = int((pred_bin & gt_bin).sum())
    fp = int((pred_bin & ~gt_bin).sum())
    fn = int((~pred_bin & gt_bin).sum())
 
    iou       = tp/(tp+fp+fn)     if (tp+fp+fn)>0 else float("nan")
    dice      = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else float("nan")
    recall    = tp/(tp+fn)        if (tp+fn)>0 else float("nan")
    precision = tp/(tp+fp)        if (tp+fp)>0 else float("nan")
    vs        = 1-abs(pred_bin.sum()-gt_bin.sum())/(pred_bin.sum()+gt_bin.sum()+1e-9)
 
    # 3D boundary metrics - defect classes only, size-guarded
    gt_count   = int(gt_bin.sum())
    pred_count = int(pred_bin.sum())
    if (class_idx in DEFECT_CLASSES and
            gt_count > 0 and pred_count > 0 and
            gt_count < MAX_BOUNDARY_VOXELS and
            pred_count < MAX_BOUNDARY_VOXELS):
        hd, assd = compute_3d_boundary_metrics(pred_bin, gt_bin)
    else:
        hd, assd = float("nan"), float("nan")
 
    # volume structure metrics (from volume_metrics.py)
    n_vox = int(pred_bin.sum())
    if n_vox > 0:
        labeled, n_comp = label(pred_bin)
        slices     = find_objects(labeled)
        comp_sizes = [int((labeled[s]==(i+1)).sum()) for i,s in enumerate(slices)]
        compacts   = [c for c in
                      [safe_compactness((labeled[s]==(i+1)))
                       for i,s in enumerate(slices)] if c is not None]
        continuity  = max(comp_sizes)/(n_vox+1e-9) if comp_sizes else float("nan")
        compactness = float(np.mean(compacts)) if compacts else float("nan")
    else:
        n_comp = 0; continuity = float("nan"); compactness = float("nan")
 
    return {
        "iou": iou, "dice": dice, "recall": recall,
        "precision": precision, "vs": vs,
        "hd_3d": hd, "assd_3d": assd,
        "volume_vox": n_vox, "components": n_comp,
        "continuity": continuity, "compactness": compactness,
    }

In [9]:
all_conditions = {}
 
for model_name, model_cfg in MODELS.items():
    if not os.path.exists(model_cfg["path"]):
        print(f"Skipping '{model_name}' - not found: {model_cfg['path']}")
        continue
    all_conditions[model_name] = {}
 
    for condition_name, apply_mrf in EVAL_CONDITIONS.items():
        print(f"\n{'─'*60}")
        print(f" {model_name} - {condition_name}")
        print(f"{'─'*60}")
 
        gt_vols, pred_vols, prob_vols = build_log_volumes(
            model_cfg, apply_3d_mrf=apply_mrf)
 
        log_metrics = {}
        for log_id in gt_vols:
            log_metrics[log_id] = {}
            for c, cls_name in enumerate(CLASS_NAMES):
                m = compute_volume_metrics_3d(pred_vols[log_id], gt_vols[log_id], c)
                log_metrics[log_id][cls_name] = m
 
        all_conditions[model_name][condition_name] = {
            "metrics":   log_metrics,
            "pred_vols": pred_vols,
            "gt_vols":   gt_vols,
        }
 
print("\nAll 3D evaluations complete.")


────────────────────────────────────────────────────────────
 DilatedCNN+MRF log-split - 2D only
────────────────────────────────────────────────────────────



────────────────────────────────────────────────────────────
 DilatedCNN+MRF log-split - 2D + 3D MRF
────────────────────────────────────────────────────────────


    Applying 3D MRF on Dub 9 (volume shape: (5, 64, 256, 256))...


    Applying 3D MRF on kmen9 (volume shape: (5, 64, 256, 256))...

────────────────────────────────────────────────────────────
 DilatedCNN+MRF random-split - 2D only
────────────────────────────────────────────────────────────



────────────────────────────────────────────────────────────
 DilatedCNN+MRF random-split - 2D + 3D MRF
────────────────────────────────────────────────────────────


...

    Applying 3D MRF on Dub 9 (volume shape: (5, 64, 256, 256))...


    Applying 3D MRF on kmen9 (volume shape: (5, 64, 256, 256))...

────────────────────────────────────────────────────────────
 UNet++ random-split - 2D only
────────────────────────────────────────────────────────────
  UNet++ encoder: 216/216 weights loaded



────────────────────────────────────────────────────────────
 UNet++ random-split - 2D + 3D MRF
────────────────────────────────────────────────────────────
  UNet++ encoder: 216/216 weights loaded


    Applying 3D MRF on Dub 9 (volume shape: (5, 64, 256, 256))...


    Applying 3D MRF on kmen9 (volume shape: (5, 64, 256, 256))...

All 3D evaluations complete.


In [15]:
OVERLAP_METRICS = ["iou", "dice", "recall", "precision", "vs", "hd_3d", "assd_3d"]
VOLUME_METRICS  = ["volume_vox", "components", "continuity", "compactness"]
DISPLAY_CLASSES = CLASS_NAMES[1:]   # skip Background in tables — not informative
# Background is still rendered in the Plotly 3D mesh (as outer hull via Bark trace)
 
def build_results_df(metric_keys, include_gt=False):
    """
    Build a tidy DataFrame from all_conditions for given metrics.
    If include_gt=True, prepends a Ground Truth row computed as
    self-comparison (gt vs gt) for volume metric reference.
    """
    rows = []
 
    # Ground Truth rows (top of each group)
    if include_gt:
        first_res = list(list(all_conditions.values())[0].values())[0]
        for log_id, gt_vol in first_res["gt_vols"].items():
            for c, cls_name in enumerate(CLASS_NAMES):
                m = compute_volume_metrics_3d(gt_vol, gt_vol, c)
                row = {"Model": "Ground Truth", "Condition": "—",
                       "Log": log_id, "Class": cls_name}
                for metric in metric_keys:
                    v = m[metric]
                    row[metric] = round(v, 4) if not np.isnan(v) else float("nan")
                rows.append(row)
 
    # Model rows
    for model_name, conditions in all_conditions.items():
        for condition_name, res in conditions.items():
            for log_id, log_data in res["metrics"].items():
                for cls_name, m in log_data.items():
                    row = {"Model": model_name, "Condition": condition_name,
                           "Log": log_id, "Class": cls_name}
                    for metric in metric_keys:
                        v = m[metric]
                        row[metric] = round(v, 4) if not np.isnan(v) else float("nan")
                    rows.append(row)
    return pd.DataFrame(rows)
 
 
def show_table(df, title, note=""):
    """
    Display results as pandas styled tables.
    - One table per class per log
    - Ground Truth row always at top, no highlighting applied to it
    - Highlighting among model rows only, per log group
    - components highlighted as lower=better
    - All numeric values formatted to 4 decimal places
    """
    from IPython.display import display as ipy_display
    print(f"\n{'='*80}")
    print(f"  {title}")
    if note:
        print(f"  {note}")
    print(f"{'='*80}")
 
    metric_cols   = [c for c in df.columns if c not in ["Model","Condition","Log","Class"]]
    # lower is better: boundary distances, compactness, components
    lower_better  = [m for m in metric_cols
                 if m in ["hd_3d", "assd_3d", "compactness", "components"]]
    higher_better = [m for m in metric_cols
                 if m in ["iou", "dice", "recall", "precision", "vs", "continuity"]]
 
    fmt_dict = {m: "{:.3f}" for m in metric_cols}
    fmt_dict.update({"volume_vox": "{:.0f}", "components": "{:.0f}"})
 
    for cls_name in DISPLAY_CLASSES:
        sub = df[df["Class"] == cls_name].drop(columns="Class")
        if sub.empty:
            continue
        print(f"\n  ── {cls_name}")
 
        for log_id in sorted(sub["Log"].unique()):
            log_sub = sub[sub["Log"] == log_id].drop(columns="Log")
 
            # split GT from models — GT always first, no sorting needed
            gt_rows    = log_sub[log_sub["Model"] == "Ground Truth"]
            model_rows = (log_sub[log_sub["Model"] != "Ground Truth"]
                          .sort_values(["Model","Condition"])
                          .reset_index(drop=True))
 
            # recombine: GT on top, models below
            combined = pd.concat([gt_rows, model_rows]).reset_index(drop=True)
            gt_idx   = combined[combined["Model"] == "Ground Truth"].index.tolist()
            print(f"    {log_id}")
 
            def highlight_best(df_s, hb=higher_better, lb=lower_better,
                                skip=gt_idx):
                styles = pd.DataFrame("", index=df_s.index, columns=df_s.columns)
                # only highlight model rows (skip GT rows)
                model_idx = [i for i in df_s.index if i not in skip]
                if not model_idx:
                    return styles
                for col in hb:
                    if col not in df_s.columns: continue
                    vals = pd.to_numeric(df_s.loc[model_idx, col], errors="coerce")
                    if vals.notna().any():
                        styles.loc[vals.idxmax(), col] =                             "background-color: #d4edda; font-weight: bold"
                for col in lb:
                    if col not in df_s.columns: continue
                    vals = pd.to_numeric(df_s.loc[model_idx, col], errors="coerce")
                    if vals.notna().any():
                        styles.loc[vals.idxmin(), col] =                             "background-color: #d4edda; font-weight: bold"
                return styles
 
            def style_gt_row(df_s, skip=gt_idx):
                """Light grey background for GT row."""
                styles = pd.DataFrame("", index=df_s.index, columns=df_s.columns)
                for i in skip:
                    styles.loc[i, :] = "background-color: #f0f0f0; font-style: italic"
                return styles
 
            styled = (combined.style
                      .apply(highlight_best, axis=None)
                      .apply(style_gt_row,   axis=None)
                      .format(fmt_dict, na_rep="n/a")
                      .set_properties(**{"text-align": "center"})
                      .set_table_styles([
                          {"selector": "th",
                           "props": [("text-align","center"),("font-weight","bold"),
                                     ("background-color","#e8e8e8")]},
                          {"selector": "tr:hover",
                           "props": [("background-color","#fafafa")]},
                      ]))
            ipy_display(styled)
 
# build dataframes
df_overlap = build_results_df(OVERLAP_METRICS, include_gt=False)
df_volume  = build_results_df(VOLUME_METRICS,  include_gt=True)
 
show_table(
    df_overlap,
    "A) Overlap Metrics - agreement with 2D GT annotations",
    note="Note: 3D MRF may reduce overlap metrics because GT was annotated "
         "slice-by-slice without enforcing 3D consistency."
)
 
show_table(
    df_volume,
    "B) Volume Metrics - anatomical plausibility of 3D structure",
    note="3D MRF expected to improve these (fewer fragments, higher continuity)."
)
 
# save both to csv
df_overlap.to_csv(os.path.join(OUT_DIR, "3d_overlap_metrics.csv"), index=False)
df_volume.to_csv(os.path.join(OUT_DIR, "3d_volume_metrics.csv"),  index=False)
print(f"\nSaved -> {OUT_DIR}/3d_overlap_metrics.csv")
print(f"Saved -> {OUT_DIR}/3d_volume_metrics.csv")


  A) Overlap Metrics - agreement with 2D GT annotations
  Note: 3D MRF may reduce overlap metrics because GT was annotated slice-by-slice without enforcing 3D consistency.

  ── Bark
    Dub 9


,Model,Condition,iou,dice,recall,precision,vs,hd_3d,assd_3d
0,DilatedCNN+MRF log-split,2D + 3D MRF,0.703,0.826,0.924,0.747,0.894,n/a,n/a
1,DilatedCNN+MRF log-split,2D only,0.711,0.831,0.931,0.750,0.892,n/a,n/a
2,DilatedCNN+MRF random-split,2D + 3D MRF,0.667,0.801,0.953,0.690,0.840,n/a,n/a
3,DilatedCNN+MRF random-split,2D only,0.671,0.803,0.958,0.692,0.839,n/a,n/a
4,UNet++ log-split,2D + 3D MRF,0.633,0.776,0.963,0.649,0.806,n/a,n/a
5,UNet++ log-split,2D only,0.639,0.779,0.969,0.652,0.804,n/a,n/a
6,UNet++ random-split,2D + 3D MRF,0.649,0.787,0.973,0.661,0.809,n/a,n/a
7,UNet++ random-split,2D only,0.653,0.790,0.976,0.664,0.809,n/a,n/a


    kmen9


,Model,Condition,iou,dice,recall,precision,vs,hd_3d,assd_3d
0,DilatedCNN+MRF log-split,2D + 3D MRF,0.463,0.633,0.824,0.513,0.768,n/a,n/a
1,DilatedCNN+MRF log-split,2D only,0.475,0.644,0.896,0.503,0.719,n/a,n/a
2,DilatedCNN+MRF random-split,2D + 3D MRF,0.578,0.732,0.826,0.658,0.887,n/a,n/a
3,DilatedCNN+MRF random-split,2D only,0.601,0.751,0.886,0.652,0.848,n/a,n/a
4,UNet++ log-split,2D + 3D MRF,0.696,0.821,0.786,0.859,0.955,n/a,n/a
5,UNet++ log-split,2D only,0.769,0.869,0.882,0.857,0.986,n/a,n/a
6,UNet++ random-split,2D + 3D MRF,0.697,0.822,0.865,0.782,0.950,n/a,n/a
7,UNet++ random-split,2D only,0.684,0.813,0.933,0.720,0.871,n/a,n/a



  ── Wood
    Dub 9


...


  ── Crack
    Dub 9


,Model,Condition,volume_vox,components,continuity,compactness
0,Ground Truth,—,106,32,0.160,n/a
1,DilatedCNN+MRF log-split,2D + 3D MRF,1105,265,0.085,16.025
2,DilatedCNN+MRF log-split,2D only,316,55,0.199,14.694
3,DilatedCNN+MRF random-split,2D + 3D MRF,593,191,0.116,14.715
4,DilatedCNN+MRF random-split,2D only,200,52,0.210,12.281
5,UNet++ log-split,2D + 3D MRF,566,76,0.191,19.932
6,UNet++ log-split,2D only,566,76,0.191,19.932
7,UNet++ random-split,2D + 3D MRF,318,77,0.192,20.577
8,UNet++ random-split,2D only,318,77,0.192,20.577


    kmen9


,Model,Condition,volume_vox,components,continuity,compactness
0,Ground Truth,—,4062,264,0.246,32.373
1,DilatedCNN+MRF log-split,2D + 3D MRF,2727,208,0.158,26.193
2,DilatedCNN+MRF log-split,2D only,2151,174,0.176,25.956
3,DilatedCNN+MRF random-split,2D + 3D MRF,2005,340,0.047,27.297
4,DilatedCNN+MRF random-split,2D only,1451,283,0.049,19.925
5,UNet++ log-split,2D + 3D MRF,2968,104,0.190,31.269
6,UNet++ log-split,2D only,2968,104,0.190,31.269
7,UNet++ random-split,2D + 3D MRF,2378,269,0.060,21.757
8,UNet++ random-split,2D only,2378,269,0.060,21.757



Saved -> ./outputs/evaluation_3d/3d_overlap_metrics.csv
Saved -> ./outputs/evaluation_3d/3d_volume_metrics.csv


In [12]:
model_names = list(all_conditions.keys()) 

def make_mesh_traces(volume_3d, spacing=(10,1,1), defect_only=False,
                     opacity_full=0.15, opacity_defect=0.7, show_legend=True):
    """
    Build Plotly Mesh3d traces from (D,H,W) label volume.
    Bark (c=1) rendered as outer hull of entire log - same as mesh_viewer.py.
    """
    traces = []; volume_3d = volume_3d.astype(np.uint8)
    classes = [3, 4] if defect_only else list(range(1, NUM_CLASSES))
    opacity = opacity_defect if defect_only else opacity_full
 
    for c in classes:
        cls_name = CLASS_NAMES[c]
        mask     = (volume_3d != 0) if (c == 1 and not defect_only) else (volume_3d == c)
        if np.count_nonzero(mask) == 0:
            continue
        try:
            verts, faces, _, _ = measure.marching_cubes(
                mask.astype(np.float32), level=0.5, spacing=spacing)
            traces.append(go.Mesh3d(
                x=verts[:,0], y=verts[:,1], z=verts[:,2],
                i=faces[:,0], j=faces[:,1], k=faces[:,2],
                color=CLASS_COLORS_PLOTLY[c], opacity=opacity,
                name=cls_name, flatshading=True,
                lighting=dict(ambient=0.5, diffuse=0.5),
                legendgroup=f"class_{c}", showlegend=show_legend,
            ))
        except Exception as e:
            print(f"  Skipping {cls_name}: {e}")
    return traces
 
def visualise_3d_comparison(log_id, gt_vol, pred_vols_dict,
                             spacing=(10,1,1), defect_only=False):
    """
    Side-by-side Plotly 3D: GT | model1 | model2 | ...
    One subplot per column. Interactive - click legend to toggle classes.
    """
    col_titles = ["Ground Truth"] + list(pred_vols_dict.keys())
    n_cols     = len(col_titles)
    fig = make_subplots(
        rows=1, cols=n_cols,
        specs=[[{"type":"mesh3d"}]*n_cols],
        subplot_titles=col_titles,
        horizontal_spacing=0.02,
    )
    scene_cfg = dict(
        aspectmode="data",
        camera=dict(eye=dict(x=-3.2, y=-4, z=0.8)),
        xaxis_title="Depth", yaxis_title="Width", zaxis_title="Height",
    )
 
    for trace in make_mesh_traces(gt_vol, spacing, defect_only):
        fig.add_trace(trace, row=1, col=1)
    for col_i, (model_name, pred_vol) in enumerate(pred_vols_dict.items(), start=2):
        for trace in make_mesh_traces(pred_vol, spacing, defect_only, show_legend=False):
            fig.add_trace(trace, row=1, col=col_i)
 
    for col_i in range(1, n_cols+1):
        fig.update_layout(**{
            ("scene" if col_i==1 else f"scene{col_i}"): scene_cfg})
 
    tag = "Defect-only" if defect_only else "Full segmentation"
    fig.update_layout(
        title=dict(text=f"3D {tag} - {log_id}", font=dict(size=15)),
        height=600, margin=dict(l=0,r=0,b=0,t=60),
        showlegend=True, legend_itemclick="toggle",
        legend_itemdoubleclick="toggleothers", uirevision="constant",
    )
    return fig
 
 
# render figures - one per log per condition
# use 2D-only predictions for the default view (more comparable to 2D eval)
# 3D MRF versions saved separately
 
for log_id in COMMON_TEST_LOGS:
    gt_vol = list(all_conditions.values())[0]["2D only"]["gt_vols"][log_id]
 
    for condition_name in EVAL_CONDITIONS:
        pred_vols_for_log = {
            model_name: all_conditions[model_name][condition_name]["pred_vols"][log_id]
            for model_name in model_names
            if condition_name in all_conditions[model_name]
            and log_id in all_conditions[model_name][condition_name]["pred_vols"]
        }
 
        cond_tag = condition_name.replace(" ","_").replace("+","")
 
        # full segmentation
        print(f"\nRendering full 3D: {log_id} - {condition_name}")
        fig = visualise_3d_comparison(log_id, gt_vol, pred_vols_for_log)
        #display(fig)
        out = os.path.join(OUT_DIR, f"3d_full_{log_id.replace(' ','_')}_{cond_tag}.html")
        fig.write_html(out); print(f"Saved -> {out}")
 
        # defect-only
        print(f"Rendering defect-only 3D: {log_id} - {condition_name}")
        fig2 = visualise_3d_comparison(log_id, gt_vol, pred_vols_for_log, defect_only=True)
        #display(fig2)
        out2 = os.path.join(OUT_DIR, f"3d_defects_{log_id.replace(' ','_')}_{cond_tag}.html")
        fig2.write_html(out2); print(f"Saved -> {out2}")


Rendering full 3D: Dub 9 - 2D only
Saved -> ./outputs/evaluation_3d\3d_full_Dub_9_2D_only.html
Rendering defect-only 3D: Dub 9 - 2D only
Saved -> ./outputs/evaluation_3d\3d_defects_Dub_9_2D_only.html

Rendering full 3D: Dub 9 - 2D + 3D MRF
Saved -> ./outputs/evaluation_3d\3d_full_Dub_9_2D__3D_MRF.html
Rendering defect-only 3D: Dub 9 - 2D + 3D MRF
Saved -> ./outputs/evaluation_3d\3d_defects_Dub_9_2D__3D_MRF.html

Rendering full 3D: kmen9 - 2D only
Saved -> ./outputs/evaluation_3d\3d_full_kmen9_2D_only.html
Rendering defect-only 3D: kmen9 - 2D only
Saved -> ./outputs/evaluation_3d\3d_defects_kmen9_2D_only.html

Rendering full 3D: kmen9 - 2D + 3D MRF
Saved -> ./outputs/evaluation_3d\3d_full_kmen9_2D__3D_MRF.html
Rendering defect-only 3D: kmen9 - 2D + 3D MRF
Saved -> ./outputs/evaluation_3d\3d_defects_kmen9_2D__3D_MRF.html
